In [11]:
import cv2
import numpy as np

# Pfad zum Video und Output-Datei
video_path = '/Users/vibe/Documents/A_CAS AICP/04_AI for Imaging/03 Projekt_04/Data/train_jp.mov'  # Ersetze mit deinem Videopfad
output_path = 'combined_shapes.png'  # Zielbild mit allen Formen

# Video öffnen
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Fehler beim Öffnen des Videos.")
    exit()

# Parameter für Frame-Sprung
frame_skip = 25  # Nur jeden 25. Frame verarbeiten
frame_count = 0
slowdown_factor = 3  # Faktor, um das Video zu verlangsamen (z.B. 3 bedeutet dreifache Verlangsamung)

# Erhalte die Dimensionen des Videos
frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Erstelle ein leeres Bild für alle Formen (gleiche Größe wie ein Frame)
combined_shapes = np.zeros((frame_height, frame_width), dtype=np.uint8)

while True:
    ret, frame = cap.read()
    if not ret:
        break  # Beende die Verarbeitung, wenn das Video zu Ende ist
    frame_count += 1
    #if frame_count % frame_skip != 0:
        #continue  # Überspringe Frames, die nicht verarbeitet werden sollen
    if frame_count % (frame_skip * slowdown_factor) != 0:
        continue  # Überspringe Frames, die nicht verarbeitet werden sollen

    # 

    # Konvertiere das Bild in Graustufen
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Rauschentfernung mit einem Gaußschen Filter
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Führe eine Kantendetektion durch
    edges = cv2.Canny(blurred, 50, 150)

    # Finde die Konturen im Bild
    contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # Zeichne die Konturen mit Formen auf das Gesamtbild
    for contour in contours:
        # Approximiere die Kontur mit geraden Linien
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, epsilon, True)

        # Filtere Formen mit 4 Ecken (Rechtecke/Trapeze)
        if len(approx) == 4:
            cv2.drawContours(combined_shapes, [approx], 0, 255, -1)  # Weiß auf Schwarz
    
    
    # Zeige das Frame mit den erkannten Formen
    cv2.imshow("Echtzeit-Vorschau der Formen", frame)
    # Füge diese Zeile zur Verlangsamung hinzu
    #cv2.waitKey(100)  # Warte 100 Millisekunden pro Frame (entspricht ca. 10 FPS)


    # Beende mit der Taste 'q'
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Speicher das kombinierte Bild
cv2.imwrite(output_path, combined_shapes)

# Video freigeben
cap.release()
print(f"Alle Formen wurden in {output_path} gespeichert.")


Alle Formen wurden in combined_shapes.png gespeichert.


: 